# Question 1

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import galsim
import galsim.hsm
from pathlib import Path

from a6cw.ellipticity import (
    unweighted_ellipticity,
    weighted_ellipticity_raw,
    hsm_ellipticity,
    calibrate_weighted_response,
    apply_simulation_calibration,
    measure_all_ellipticities,
    plot_ellipticity_comparison,
)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

STAMPS_FILE = DATA_DIR / "galaxy_stamps.fits"

## Question 1a

Ellipticity is defined in terms of quadrupole moments by

\begin{equation}
    \epsilon_1=\frac{Q_{11}-Q_{22}}{Q_{11}+Q_{22}+2\sqrt{Q_{11}Q_{22}-Q_{12}^2}},
\end{equation}

\begin{equation}
    \epsilon_2=\frac{2 Q_{12}}{Q_{11}+Q_{22}+2\sqrt{Q_{11}Q_{22}-Q_{12}^2}},
\end{equation}

where

\begin{equation}
    Q_{pq}=\frac{\int dxdyI(x,y)(p-\bar{p})(q-\bar{q})}{\int dxdyI(x,y)},
\end{equation}

and

\begin{equation}
    \bar{x}=\frac{\int dxdy I(x,y) x}{\int dxdy I(x,y)}.
\end{equation}

These equations can be modified to include the effects of weighting the pixels. The expressions for ellipticity are unchanged, but the quadrupole moments become

\begin{equation}
    Q^W_{ij}=\frac{\int dxdy W(x,y)I(x,y)(p-\bar{p})(q-\bar{q})}{\int dxdy W(x,y)I(x,y)},
\end{equation}

and

\begin{equation}
    \bar{p}=\frac{\int dxdy W(x,y) I(x,y) p}{\int dxdy W(x,y) I(x,y)}.
\end{equation}

For the weight function, we use a circular Gaussian:

\begin{equation}
    W(x,y)=\exp(-\frac{1}{2\sigma^2}((x-\bar{x})^2+(y-\bar{y})^2)),
\end{equation}

where the standard deviation is related to the FWHM by

\begin{equation}
    \text{FWHM}=2\sqrt{2\ln2}\sigma.
\end{equation}

In [2]:
def main():
    results = measure_all_ellipticities(STAMPS_FILE) # Measure ellipticities for all galaxy stamps using the defined estimators
    plot_ellipticity_comparison(results, OUTPUT_DIR) # Generate diagnostic plots comparing the ellipticity estimates from different methods
 
    # Save for use in Q2
    np.save(OUTPUT_DIR / "ellipticities_unweighted.npy", results["unweighted"]) # Save unweighted ellipticity results to a .npy file in the outputs directory
    np.save(OUTPUT_DIR / "ellipticities_weighted.npy", results["weighted"]) # Save weighted ellipticity results to a .npy file in the outputs directory
    np.save(OUTPUT_DIR / "ellipticities_hsm.npy", results["hsm"]) # Save HSM ellipticity results to a .npy file in the outputs directory
    print("\nEllipticity arrays saved to outputs/.")
    return results
 
 
if __name__ == "__main__":
    main()

Loaded 100000 galaxy stamps from data/galaxy_stamps.fits



Median HSM galaxy size: sigma_g = 3.436 px

Summary statistics:
  unweighted  :  92351/100000 valid | <e1> = -0.0002  σ(e1) = 0.1047 | <e2> = +0.0001  σ(e2) = 0.1353
  weighted    :  99997/100000 valid | <e1> = +0.0000  σ(e1) = 0.0116 | <e2> = +0.0000  σ(e2) = 0.0116
  hsm         : 100000/100000 valid | <e1> = +0.0001  σ(e1) = 0.0572 | <e2> = +0.0001  σ(e2) = 0.0573


Saved outputs/q1_scatter_e1_e2.png


Saved outputs/q1_histograms.png


Saved outputs/q1_comparison_vs_hsm.png

Force-through-origin regression slopes (method vs HSM):
  unweighted_e1            :  R = 1.0240 +/- 0.0050
  unweighted_e2            :  R = 1.0091 +/- 0.0071
  weighted_e1              :  R = 0.1974 +/- 0.0001
  weighted_e2              :  R = 0.1974 +/- 0.0001

Ellipticity arrays saved to outputs/.
